# Headline networks

Associated chart: the who-Fox-names-together network scrolly and the headline table

Output files:
`headline_network.json`, `article_network.json`, and `fox_player_headlines.csv` in `src/data/`.

Input files: `data/fox_wnba.csv`.

The cast of tracked people — names, roles, and their disambiguation patterns — is the `HN_PEOPLE` table below; edit it there.


## Setup

In [ ]:
import csv, json, re
from collections import Counter, defaultdict
from itertools import combinations
from pathlib import Path

DATA = Path("../data")
OUT = Path("../src/data")

csv.field_size_limit(10**8)  # article bodies overflow the default field limit
fox_union = {}  # url -> {date, title, body}, deduped by URL, first row wins
for r in csv.DictReader(open(DATA / "fox_wnba.csv")):
    u = (r.get("url") or "").rstrip("/")
    if u and u not in fox_union:
        fox_union[u] = {"date": r.get("date", ""), "title": r.get("headline", ""),
                        "body": r.get("body", "")}
print(f"{len(fox_union):,} Fox WNBA articles")

## The cast

Every person the networks track: `name -> (group, regex)`. Groups are
`players` / `politics` / `media` / `league`. Edit this dict to add or drop people —
everything below just reruns. Comments mark the regexes that need care (shared
surnames, non-WNBA figures).

Who is in: everyone Fox News named in 15 or more WNBA headlines (counted by name match with the regex here — the NER counts in `data/fox_people_mentions.csv` overstate people who share a surname with someone bigger, e.g. Cherelle Griner), plus a named cast of political and media figures kept for the argument regardless of count (Kirk, Vance, Portnoy, Travis, Rowling, Stephen A. Smith, Hostin, Wallen, Lia Thomas, the Obamas).


In [ ]:
# canonical roster: name -> (group, regex), from people.json
# people flagged "network": false are in the roster for the framing view
# only: they are matched in the headline table (ALL_RX) but get no node
ROSTER = json.loads((OUT / "people.json").read_text())
HN_PEOPLE = {name: (p["group"], p["rx"]) for name, p in ROSTER.items() if p.get("network", True)}
HN_RX = {name: re.compile(pat, re.I) for name, (grp, pat) in HN_PEOPLE.items()}
ALL_RX = {name: re.compile(p["rx"], re.I) for name, p in ROSTER.items()}
HN_YEARS = ["2019", "2020", "2021", "2022", "2023", "2024", "2025", "2026"]

# Some surnames belong to more than one person in this corpus (Alysha Clark vs
# Caitlin Clark). If the look-alike phrase appears and the person's own first
# name does not, the match is rejected.
PERSON_LOOKALIKES = {
    "Caitlin Clark": (re.compile(r"alysha\W{0,3}clark", re.I), re.compile(r"caitlin", re.I)),
    # a bare "Obama" in a Michelle Obama headline is not Barack
    "Barack Obama": (re.compile(r"michelle obama", re.I), re.compile(r"barack", re.I)),
    # a bare "Griner" in a Cherelle Griner headline is not Brittney
    "Brittney Griner": (re.compile(r"cherelle\W{0,3}griner", re.I), re.compile(r"brittney", re.I)),
    "Sophie Cunningham": (re.compile(r"cade\W{0,3}cunningham", re.I), re.compile(r"sophie", re.I)),
    "Breanna Stewart": (re.compile(r"isaiah\W{0,3}stewart", re.I), re.compile(r"breanna", re.I)),
    "Angel Reese": (re.compile(r"arvell\W{0,3}reese", re.I), re.compile(r"angel", re.I)),
}

def person_match(name, rx, text, context=None):
    """Does `text` (a headline) name this person? `context` is the article
    body: a look-alike named anywhere in the article ("Taurasi re-signs …
    while Clark heads to Washington", about Alysha Clark) rules the bare
    surname out unless the headline carries the person's own first name."""
    if not rx.search(text):
        return False
    rule = PERSON_LOOKALIKES.get(name)
    if rule:
        lookalike, required = rule
        if (lookalike.search(text) or (context and lookalike.search(context))) \
                and not required.search(text):
            return False
    return True

print(len(HN_PEOPLE), "people tracked")

## Sentence evidence helpers

ALL-CAPS links to other stories mid-article are stripped.


In [ ]:
# a run of four or more shouted words is a pasted link, not a sentence;
# ordinary acronyms (WNBA, ESPN, US) survive
SHOUTED_RUN = re.compile(r"\b(?:[A-Z][A-Z'\u2019\u2018\u2019.&-]{1,}[,:;?!]?\s+){3,}[A-Z][A-Z'\u2019\u2018\u2019.&-]{1,}[,:;?!]?")

def strip_shouting(sentence):
    return re.sub(r"\s{2,}", " ", SHOUTED_RUN.sub(" ", sentence)).strip(" \u2014-\u2013\u2014,;:")

def names_in_sentence(sentence, names):
    """Which of `names` this sentence refers to.

    `names` are people already known to be in the article, so a bare surname is
    safe — unless two of them share one (Lia and Alyssa Thomas), in which case
    only the full pattern counts. Capitalized-only surname matching separates
    the person ("Freedom wrapped up") from the concept ("religious freedom").
    """
    surname_counts = Counter(name.split()[-1] for name in names)
    found = []
    for name in names:
        surname = name.split()[-1]
        hit = HN_RX[name].search(sentence)
        if not hit and len(surname) > 3 and surname_counts[surname] == 1:
            hit = re.search(r"\b" + re.escape(surname) + r"\b", sentence)
        if not hit:
            continue
        rule = PERSON_LOOKALIKES.get(name)
        if rule:
            lookalike, required = rule
            if lookalike.search(sentence) and not required.search(sentence):
                continue
        found.append(name)
    return found

MAX_SENTENCE_CHARS = 700  # Fox runs long, quote-stuffed sentences; keep but trim
DISPLAY_CHARS = 240

def trim_sentence(sentence):
    if len(sentence) <= DISPLAY_CHARS:
        return sentence
    cut = sentence[:DISPLAY_CHARS].rsplit(" ", 1)[0]
    return cut.rstrip(",;:\u2014 ") + "\u2026"

def person_sentences(body, names, limit=2):
    """Sentences naming exactly one of `names` — the fallback for links where
    Fox pairs two people in a headline but never in a single body sentence."""
    alone = {name: [] for name in names}
    if not body:
        return alone
    for sentence in re.split(r"(?<=[.!?])\s+", body.replace("\n", " ")):
        sentence = strip_shouting(sentence.strip())
        if not (40 <= len(sentence) <= MAX_SENTENCE_CHARS):
            continue
        named = names_in_sentence(sentence, names)
        if len(named) == 1 and len(alone[named[0]]) < limit:
            alone[named[0]].append(trim_sentence(sentence))
    return alone

def shared_sentences(body, names, limit=3):
    """Sentences that name two or more of `names`: {(a, b): [sentence, ...]} —
    the evidence behind a link in the article network."""
    together = {}
    if len(names) < 2 or not body:
        return together
    for sentence in re.split(r"(?<=[.!?])\s+", body.replace("\n", " ")):
        sentence = strip_shouting(sentence.strip())
        if not (40 <= len(sentence) <= MAX_SENTENCE_CHARS):
            continue
        named = sorted(names_in_sentence(sentence, names))
        if len(named) < 2:
            continue
        trimmed = trim_sentence(sentence)
        for a, b in combinations(named, 2):
            bucket = together.setdefault((a, b), [])
            if len(bucket) < limit and trimmed not in bucket:
                bucket.append(trimmed)
    return together

def evidence_for(pair, both_named, singly_named, limit=3):
    """Best sentences for one link: sentences naming both people, else one
    sentence about each, so every headline still carries some context."""
    a, b = pair
    if both_named.get(pair):
        return both_named[pair][:limit]
    one_each = [s[0] for s in (singly_named.get(a), singly_named.get(b)) if s]
    return one_each if len(set(one_each)) == len(one_each) else one_each[:1]

## Headline network

Two people are linked when Fox names them in the SAME headline.


In [ ]:
hn_solo = {name: Counter() for name in HN_PEOPLE}       # name -> year -> headlines
hn_solo_ex = {name: {} for name in HN_PEOPLE}           # name -> year -> a headline
hn_edges = defaultdict(lambda: defaultdict(list))       # year -> (a,b) -> [headlines]
hn_edges_mo = defaultdict(lambda: defaultdict(int))     # month -> (a,b) -> count
hn_solo_mo = {name: Counter() for name in HN_PEOPLE}    # name -> month -> headlines
hn_own = {name: [] for name in HN_PEOPLE}               # name -> [{d, t}] every headline

for record in fox_union.values():
    year = (record.get("date") or "")[:4]
    headline = (record.get("title") or "").strip()
    if year not in HN_YEARS or not headline:
        continue
    month = (record.get("date") or "")[:7]
    present = [name for name, rx in HN_RX.items()
               if person_match(name, rx, headline, record.get("body") or "")]
    for name in present:
        hn_solo[name][year] += 1
        hn_solo_mo[name][month] += 1
        hn_solo_ex[name].setdefault(year, headline)
        hn_own[name].append({"d": (record.get("date") or "")[:10], "t": headline})
    for a, b in combinations(sorted(present), 2):
        hn_edges[year][(a, b)].append({"t": headline, "s": []})
        hn_edges_mo[month][(a, b)] += 1

hn_nodes = [{"name": name, "group": HN_PEOPLE[name][0],
             "counts": [hn_solo[name].get(y, 0) for y in HN_YEARS],
             "total": sum(hn_solo[name].values()),
             "examples": hn_solo_ex[name],
             # oldest first, so early-year views have something to show
             "headlines": sorted(hn_own[name], key=lambda h: h["d"])}
            for name in HN_PEOPLE if sum(hn_solo[name].values()) > 0]
hn_edges_out = {
    y: [{"a": a, "b": b, "n": len(hs), "ex": hs[0]["t"],
         "heads": [h["t"] for h in hs[:40]]}
        for (a, b), hs in sorted(hn_edges[y].items(), key=lambda kv: -len(kv[1]))]
    for y in HN_YEARS if hn_edges.get(y)
}
# dense month axis so the animation can grow smoothly instead of jumping years
hn_months = sorted({m for m in hn_edges_mo} | {m for n in hn_solo_mo.values() for m in n})
hn_months = [m for m in hn_months if len(m) == 7 and m[:4] in HN_YEARS]
hn_edges_mo_out = {m: [[a, b, n] for (a, b), n in sorted(hn_edges_mo[m].items(), key=lambda kv: -kv[1])]
                   for m in hn_months if hn_edges_mo.get(m)}
hn_solo_mo_out = {name: {m: c for m, c in hn_solo_mo[name].items() if m in hn_months}
                  for name in HN_PEOPLE if hn_solo_mo[name]}

#### Overwrites `src/data/headline_network.json`

In [ ]:
json.dump({"years": HN_YEARS, "months": hn_months, "nodes": hn_nodes,
           "edges": hn_edges_out, "edges_monthly": hn_edges_mo_out,
           "solo_monthly": hn_solo_mo_out}, open(OUT / "headline_network.json", "w"))
print("headline_network:", {y: len(e) for y, e in hn_edges_out.items()})

## Article network

Same people and shape, but two people are linked when they appear anywhere in the
same ARTICLE. Each edge carries the sentences naming them together (`sents`).


In [ ]:
an_solo = {name: Counter() for name in HN_PEOPLE}
an_solo_mo = {name: Counter() for name in HN_PEOPLE}
an_solo_ex = {name: {} for name in HN_PEOPLE}
an_edges = defaultdict(lambda: defaultdict(list))
an_edges_mo = defaultdict(lambda: defaultdict(int))
an_own = {name: [] for name in HN_PEOPLE}

for record in fox_union.values():
    year = (record.get("date") or "")[:4]
    month = (record.get("date") or "")[:7]
    headline = (record.get("title") or "").strip()
    if year not in HN_YEARS or not headline:
        continue
    text = (record.get("title") or "") + " " + (record.get("body") or "")
    present = [name for name, rx in HN_RX.items() if person_match(name, rx, text)]
    for name in present:
        an_solo[name][year] += 1
        an_solo_mo[name][month] += 1
        an_solo_ex[name].setdefault(year, headline)
        an_own[name].append({"d": (record.get("date") or "")[:10], "t": headline})
    body_text = record.get("body") or ""
    pair_sentences = shared_sentences(body_text, present)
    solo_sentences = person_sentences(body_text, present)
    for a, b in combinations(sorted(present), 2):
        an_edges[year][(a, b)].append(
            {"t": headline, "s": evidence_for((a, b), pair_sentences, solo_sentences)})
        an_edges_mo[month][(a, b)] += 1

an_nodes = [{"name": name, "group": HN_PEOPLE[name][0],
             "counts": [an_solo[name].get(y, 0) for y in HN_YEARS],
             "total": sum(an_solo[name].values()),
             "examples": an_solo_ex[name],
             "headlines": sorted(an_own[name], key=lambda h: h["d"])}
            for name in HN_PEOPLE if sum(an_solo[name].values()) > 0]
an_months = sorted({m for m in an_edges_mo} | {m for n in an_solo_mo.values() for m in n})
an_months = [m for m in an_months if len(m) == 7 and m[:4] in HN_YEARS]

#### Overwrites `src/data/article_network.json`

In [ ]:
json.dump({"years": HN_YEARS, "months": an_months, "nodes": an_nodes,
           "edges": {y: [{"a": a, "b": b, "n": len(hs), "ex": hs[0]["t"],
                          "heads": [h["t"] for h in hs[:60]],
                          "sents": {h["t"]: h["s"] for h in hs[:60] if h["s"]}}
                         for (a, b), hs in sorted(an_edges[y].items(), key=lambda kv: -len(kv[1]))]
                     for y in HN_YEARS if an_edges.get(y)},
           "edges_monthly": {m: [[a, b, n] for (a, b), n in sorted(an_edges_mo[m].items(), key=lambda kv: -kv[1])]
                             for m in an_months if an_edges_mo.get(m)},
           "solo_monthly": {name: {m: c for m, c in an_solo_mo[name].items() if m in an_months}
                            for name in HN_PEOPLE if an_solo_mo[name]}},
          open(OUT / "article_network.json", "w"))
print("article_network:", {y: len(e) for y, e in an_edges.items()})

## Player headlines table

Every Fox WNBA headline, tagged with whichever tracked people it names.


In [ ]:
table_rows = []
for url, record in fox_union.items():
    headline = (record.get("title") or "").strip()
    date = (record.get("date") or "")[:10]
    if not headline or not date:
        continue
    who = [name for name, rx in ALL_RX.items()
           if person_match(name, rx, headline, record.get("body") or "")]
    table_rows.append({"d": date, "t": headline, "p": who, "u": url})
table_rows.sort(key=lambda r: r["d"])

#### Overwrites `src/data/fox_player_headlines.csv`

In [ ]:
with open(OUT / "fox_player_headlines.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["date", "title", "players", "url"])
    for r in table_rows:
        w.writerow([r["d"], r["t"], "; ".join(r["p"]), r["u"]])
named = sum(1 for r in table_rows if r["p"])
print(f"fox_player_headlines: {len(table_rows)} headlines ({named} name someone tracked)")

## Preview: top headline pairings of 2026


In [ ]:
for e in hn_edges_out["2026"][:10]:
    print(f"{e['n']:>3}  {e['a']} \u2014 {e['b']}")

## Spot check

In [ ]:
import random

year = random.choice([y for y in hn_edges if hn_edges[y]])
print(f"— random {year} headline edges (both people share one headline)")
for (a, b), hs in random.sample(list(hn_edges[year].items()), min(4, len(hn_edges[year]))):
    print(f"   {a} — {b} ({len(hs)}): {hs[0]['t'][:70]}")

year = random.choice([y for y in an_edges if an_edges[y]])
pair, hs = random.choice(list(an_edges[year].items()))
print(f"\n— one random {year} article edge, with its evidence sentence")
print(f"   {pair[0]} — {pair[1]} ({len(hs)} articles)")
withsent = [h for h in hs if h.get("s")]
if withsent:
    h = random.choice(withsent)
    print(f"   “{h['s'][:150]}”")